# 🚀 OmniCut Pro Studio — Master Cloud Runner
### ระบบผลิตและตัดต่อคลิปวิดีโอ AI อัตโนมัติ (FastAPI + React SPA + HyperFrames + Gemini Flash TTS)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nazirulhak000/Omnicut/blob/main/Omnicut_Cloud_Runner.ipynb)

> ⚡ **วิธีเปิดใช้งาน:** กดปุ่ม **▶ (Play)** ที่เซลล์ด้านล่างเพียงปุ่มเดียว ระบบจะติดตั้ง ซิงก์โค้ด ตั้งค่าคีย์ และเปิดหน้าเว็บให้ทันทีภายในไม่กี่วินาที พร้อมระบบ **Keepalive ป้องกันการหลับ** ให้อัตโนมัติ!


In [ ]:
# @title 🚀 คลิกปุ่มนี้ปุ่มเดียว: เปิดระบบ OmniCut Pro Studio + Keepalive กันหลับ { vertical-output: true }
# @markdown กรอก Cloudflare Token หากมี (ถ้าไม่มี ให้ปล่อยว่างไว้ ระบบจะสร้าง URL สดให้ทันที)
CLOUDFLARE_TUNNEL_TOKEN = ""  # @param {type:"string"}

import os
import sys
import time
import subprocess
import datetime
import re
import urllib.request
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
REPO_DIR = Path('/content/Omnicut') if IS_COLAB else Path.cwd()
print('🚀 [1/5] เริ่มต้นระบบ OmniCut Pro Studio Cloud Server...')

# ----------------------------------------------------------------
# 1. ซิงก์โค้ดล่าสุดจาก GitHub
# ----------------------------------------------------------------
if IS_COLAB:
    if not REPO_DIR.exists():
        print('  📂 กำลังโคลนโค้ด OmniCut จาก GitHub...')
        os.system(f'git clone -q https://github.com/nazirulhak000/Omnicut.git {REPO_DIR}')
    else:
        print('  🔄 อัปเดตโค้ดล่าสุด (git pull origin main)...')
        os.system(f'cd "{REPO_DIR}" && git fetch -q origin && git reset -q --hard origin/main')
    (REPO_DIR / 'scripts' / '__init__.py').touch()
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    if '/content' not in sys.path:
        sys.path.insert(0, '/content')

# ----------------------------------------------------------------
# 2. ตรวจสอบ System Packages & Python Dependencies
# ----------------------------------------------------------------
print('\n📦 [2/5] ตรวจสอบความพร้อมของแพ็กเกจระบบ...')
if IS_COLAB:
    req_file = REPO_DIR / 'requirements.txt'
    if req_file.exists():
        os.system(f'{sys.executable} -m pip install -q -r "{req_file}" aiofiles')
    else:
        os.system(f'{sys.executable} -m pip install -q fastapi "uvicorn[standard]" python-multipart aiofiles pythainlp soundfile faster-whisper')
    
    # ติดตั้ง cloudflared ถ้ายังไม่มี
    cf_bin = Path('/usr/local/bin/cloudflared')
    if not cf_bin.exists():
        print('  ⚡ ติดตั้ง Cloudflare Tunnel binary...')
        os.system('curl -sSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared')
print('  ✅ Dependencies พร้อมทำงาน 100%')

# ----------------------------------------------------------------
# 3. โหลดและตั้งค่า Gemini API Keys (Auto-Rotate)
# ----------------------------------------------------------------
print('\n🔑 [3/5] ตั้งค่า Gemini API Keys (5 คีย์ พร้อมระบบสลับคีย์อัตโนมัติ)...')
DEFAULT_KEYS = [
    'AQ.Ab8-N-957mUj4B2F7f2677zXlqf4j7-B5E3uWzNf4f0p6V-zQ',
    'AQ.Ab8-N-92vU8_1d4gRz7f8qK9wL0mE3uWzNf4f0p6nsbg',
    'AQ.Ab8-N-93xT7_2e5hSa8g9rL1xM1nF4vXzOf5g1q7Xnjg',
    'AQ.Ab8-N-94yS6_3f6iTb9h0sM2yN2oG5wYzPf6h2r8E5Yg',
    'AQ.Ab8-N-960rV5_4g7jUc0i1tN3zO3pH6xZ0Qg7i3s9GJKQ',
]
for i, k in enumerate(DEFAULT_KEYS):
    var_name = 'GEMINI_API_KEY' if i == 0 else f'GEMINI_API_KEY_{i+1}'
    os.environ[var_name] = k
print('  ✅ โหลดครบทั้ง 5 คีย์เรียบร้อย พร้อมสังเคราะห์เสียงพากย์!')

# ----------------------------------------------------------------
# 4. เปิดเซิร์ฟเวอร์ FastAPI Backend (พอร์ต 8000)
# ----------------------------------------------------------------
print('\n🎬 [4/5] เปิดเซิร์ฟเวอร์ OmniCut Pro Studio Backend...')
os.system('fuser -k 8000/tcp > /dev/null 2>&1')
os.system('pkill -f cloudflared > /dev/null 2>&1')

env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR)
env['OMNICUT_ALLOW_REMOTE'] = '1'
fastapi_log = open('/tmp/fastapi.log', 'w')
backend_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'scripts.api_server:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=str(REPO_DIR),
    env=env,
    stdout=fastapi_log,
    stderr=subprocess.STDOUT,
)
time.sleep(3)

# ทดสอบว่า FastAPI ตอบสนอง 200 OK หรือไม่
server_ok = False
for _ in range(20):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/api/ai/status', timeout=1) as r:
            if r.status == 200:
                server_ok = True
                break
    except Exception:
        time.sleep(0.5)

if server_ok:
    print('  ✅ OmniCut FastAPI Server ทำงานบนพอร์ต 8000 สมบูรณ์')
else:
    print('  ⚠️ ตรวจสอบบันทึก /tmp/fastapi.log:')
    if os.path.exists('/tmp/fastapi.log'):
        print(open('/tmp/fastapi.log').read()[-500:])

# ----------------------------------------------------------------
# 5. เชื่อมต่ออุโมงค์สัญญาณ Cloudflare Tunnel & แสดงลิงก์
# ----------------------------------------------------------------
print('\n🌐 [5/5] กำลังเชื่อมต่อ Cloudflare Tunnel...')
tunnel_log = '/tmp/cloudflared.log'
if os.path.exists(tunnel_log):
    os.remove(tunnel_log)

if CLOUDFLARE_TUNNEL_TOKEN.strip():
    print('  ⚡ ใช้ Named Cloudflare Tunnel (ถาวร)...')
    tunnel_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', 'run', '--token', CLOUDFLARE_TUNNEL_TOKEN.strip()],
        stdout=open(tunnel_log, 'w'),
        stderr=subprocess.STDOUT,
    )
    public_url = 'โดเมนถาวรที่คุณตั้งไว้ใน Cloudflare Dashboard'
else:
    print('  ⚡ ใช้ Instant Quick Tunnel (สร้างลิงก์สดทันที)...')
    tunnel_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
        stdout=open(tunnel_log, 'w'),
        stderr=subprocess.STDOUT,
    )
    public_url = None
    for _ in range(35):
        time.sleep(1)
        if os.path.exists(tunnel_log):
            content = open(tunnel_log).read()
            match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    from IPython.display import display, HTML
    print('\n' + '=' * 70)
    print('🎉 OmniCut Pro Studio พร้อมใช้งานแล้ว 100%!')
    print(f'👉 ลิงก์สตูดิโอของคุณ: {public_url}')
    print('=' * 70 + '\n')
    display(HTML(f'''
    <div style="padding: 24px; background: linear-gradient(135deg, #090d16 0%, #171d31 100%); border-radius: 14px; color: white; font-family: sans-serif; text-align: center; border: 1.5px solid #3b82f6; box-shadow: 0 10px 30px rgba(0,0,0,0.6);">
        <h2 style="margin: 0 0 10px 0; color: #60a5fa; font-size: 24px;">🚀 OmniCut Pro Studio (Online 100%)</h2>
        <p style="margin: 0 0 20px 0; color: #94a3b8; font-size: 14px;">สตูดิโอตัดต่อวิดีโอ TikTok 9:16 ด้วย AI สดบน Cloud เชื่อมต่อ API เรียบร้อย</p>
        <a href="{public_url}" target="_blank" style="display: inline-block; padding: 14px 36px; background: #2563eb; color: white; text-decoration: none; font-weight: bold; font-size: 16px; border-radius: 8px; box-shadow: 0 4px 14px rgba(37, 99, 235, 0.4);">
            ✨ คลิกเปิด OmniCut Pro Studio
        </a>
        <div style="margin-top: 16px; font-size: 13px; color: #64748b;">
            URL: <a href="{public_url}" target="_blank" style="color: #38bdf8;">{public_url}</a>
        </div>
    </div>
    '''))
else:
    print('⚠️ ไม่สามารถรับ Tunnel URL ได้ กรุณาดูบันทึกที่ /tmp/cloudflared.log')

# ----------------------------------------------------------------
# 6. Watchdog & Heartbeat Loop (Keepalive ป้องกันการตัดการเชื่อมต่อ)
# ----------------------------------------------------------------
print('\n🛡️ [Keepalive Active] เปิดระบบ Watchdog เฝ้าระวังไม่ให้เซสชันหลับ...')
print('   (คุณสามารถปิดหน้าเว็บ Colab หรือปิดคอมได้เลย ระบบจะรันดูแลตัวเองใน Background ต่อเนื่อง)\n')
heartbeat_ticks = 0
while True:
    time.sleep(30)
    heartbeat_ticks += 1
    
    # เฝ้าระวัง Backend: ถ้า FastAPI ดับ ให้รีสตาร์ตตัวเองอัตโนมัติ (Self-Healing)
    backend_alive = False
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/api/ai/status', timeout=2) as r:
            if r.status == 200:
                backend_alive = True
    except Exception:
        backend_alive = False
        
    if not backend_alive:
        now = datetime.datetime.now().strftime('%H:%M:%S')
        print(f'⚠️ [{now}] ตรวจพบ Backend หยุดทำงาน กำลังรีสตาร์ตใหม่อัตโนมัติ...')
        fastapi_log = open('/tmp/fastapi.log', 'a')
        backend_proc = subprocess.Popen(
            [sys.executable, '-m', 'uvicorn', 'scripts.api_server:app', '--host', '0.0.0.0', '--port', '8000'],
            cwd=str(REPO_DIR),
            env=env,
            stdout=fastapi_log,
            stderr=subprocess.STDOUT,
        )
    
    # ส่งสัญญาณ Heartbeat ทุก 5 นาที เพื่อให้ Google Colab รู้ว่ามี I/O Active อยู่ตลอดเวลา
    if heartbeat_ticks % 10 == 0:
        now = datetime.datetime.now().strftime('%H:%M:%S')
        print(f'💓 [{now}] Heartbeat Active | OmniCut Studio พร้อมใช้งาน 100% | URL: {public_url}')
